# Análise dos Indicadores de Desenvolvimento Global (WDI)
## Período: 2019-2023 (Pandemia COVID-19)

---

### Objetivo
Analisar o comportamento dos países durante o período da pandemia COVID-19 utilizando:
- Técnicas de limpeza e normalização de dados
- Feature Engineering
- Redução de dimensionalidade via PCA (implementação própria)
- Visualização através de gráfico de dispersão

## 1. Importação de Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import textwrap
from pathlib import Path
from IPython.display import display, HTML
from scipy import stats


# Configurações de visualização
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10
sns.set_style('whitegrid')

# Suprimir avisos
import warnings
warnings.filterwarnings('ignore')

## 2. Carregamento e Exploração Inicial dos Dados

In [ ]:
# Carregar datasets
df_main = pd.read_csv('WDICSV.csv')
df_series = pd.read_csv('WDISeries.csv')
df_country = pd.read_csv('WDICountry.csv')

print("Dimensões do dataset principal:", df_main.shape)
print("\nPrimeiras linhas:")
df_main.head()

In [ ]:
# Explorar estrutura dos dados
print("Informações do dataset:")
print(df_main.info())
print("\nColunas disponíveis:")
print(df_main.columns.tolist())

## 3. Seleção de Features por Pilar

Selecionaremos 12-18 features distribuídas entre os 4 pilares, com atenção especial para:
- Relevância para análise do período pandêmico
- Volume de dados faltantes (< 40%)
- Cobertura temporal 2019-2023

In [ ]:
# Dicionário de features selecionadas por pilar
features_selecionadas = {
    # Pilar Econômico
    'PIB_per_capita': 'NY.GDP.PCAP.CD',
    'Inflacao': 'FP.CPI.TOTL.ZG',
    'Exportacoes_PIB': 'NE.EXP.GNFS.ZS',
    'Desemprego': 'SL.UEM.TOTL.ZS',
    
    # Pilar de Saúde
    'Expectativa_Vida': 'SP.DYN.LE00.IN',
    'Mortalidade_Infantil': 'SP.DYN.IMRT.IN',
    'Gasto_Saude_Per_Capita': 'SH.XPD.CHEX.PC.CD',
    'Leitos_Hospitalares': 'SH.MED.BEDS.ZS',
    
    # Pilar Educação/Social
    'Taxa_Alfabetizacao': 'SE.ADT.LITR.ZS',
    'Acesso_Internet': 'IT.NET.USER.ZS',
    'Populacao_Urbana': 'SP.URB.TOTL.IN.ZS',
    'Acesso_Eletricidade': 'EG.ELC.ACCS.ZS',
    
    # Pilar Ambiental
    'Emissoes_CO2_per_capita': 'EN.ATM.CO2E.PC',
    'Consumo_Energia': 'EG.USE.PCAP.KG.OE',
    'Energia_Renovavel': 'EG.FEC.RNEW.ZS',
    
    # Features auxiliares
    'Populacao_Total': 'SP.POP.TOTL'
}

print(f"Total de features selecionadas: {len(features_selecionadas)}")
print("\nFeatures por pilar:")
print("- Econômico: 4")
print("- Saúde: 4")
print("- Educação/Social: 4")
print("- Ambiental: 3")
print("- Auxiliar: 1")

## 4. Extração e Pivotamento dos Dados

In [ ]:
# Filtrar período 2019-2023
anos_analise = ['2019', '2020', '2021', '2022', '2023']
colunas_anos = [col for col in df_main.columns if col in anos_analise]

# Filtrar apenas indicadores selecionados
codigos_indicadores = list(features_selecionadas.values())
df_filtrado = df_main[df_main['Indicator Code'].isin(codigos_indicadores)].copy()

print(f"Dataset filtrado: {df_filtrado.shape}")
print(f"Países únicos: {df_filtrado['Country Code'].nunique()}")
print(f"Indicadores únicos: {df_filtrado['Indicator Code'].nunique()}")

In [ ]:
# Criar dicionário reverso
codigo_para_nome = {v: k for k, v in features_selecionadas.items()}

# Pivotar dados: uma linha por país, colunas para cada indicador
# Calcular média do período 2019-2023 para cada indicador
df_pivot_list = []

for pais in df_filtrado['Country Code'].unique():
    dados_pais = df_filtrado[df_filtrado['Country Code'] == pais]
    registro = {'Country_Code': pais}
    
    # Nome do país
    registro['Country_Name'] = dados_pais['Country Name'].iloc[0]
    
    # Para cada indicador, calcular média do período
    for codigo in codigos_indicadores:
        dados_indicador = dados_pais[dados_pais['Indicator Code'] == codigo]
        if len(dados_indicador) > 0:
            valores = dados_indicador[colunas_anos].values.flatten()
            valores_numericos = pd.to_numeric(valores, errors='coerce')
            media = np.nanmean(valores_numericos)
            nome_feature = codigo_para_nome.get(codigo, codigo)
            registro[nome_feature] = media
        else:
            nome_feature = codigo_para_nome.get(codigo, codigo)
            registro[nome_feature] = np.nan
    
    df_pivot_list.append(registro)

df_analise = pd.DataFrame(df_pivot_list)

print("Dataset pivotado criado:")
print(df_analise.shape)
df_analise.head()

## 5. Análise de Dados Faltantes

In [ ]:
# Calcular percentual de valores faltantes por coluna
missing_pct = (df_analise.isnull().sum() / len(df_analise) * 100).sort_values(ascending=False)

print("Percentual de valores faltantes por feature:\n")
for col, pct in missing_pct.items():
    if col not in ['Country_Code', 'Country_Name']:
        print(f"{col:30s}: {pct:6.2f}%")

# Visualizar dados faltantes
plt.figure(figsize=(12, 6))
features_numericas = [col for col in df_analise.columns if col not in ['Country_Code', 'Country_Name']]
missing_data = (df_analise[features_numericas].isnull().sum() / len(df_analise) * 100).sort_values(ascending=True)

plt.barh(range(len(missing_data)), missing_data.values)
plt.yticks(range(len(missing_data)), missing_data.index)
plt.xlabel('Percentual de Dados Faltantes (%)')
plt.title('Análise de Dados Faltantes por Feature')
plt.axvline(x=40, color='r', linestyle='--', label='Limite 40%')
plt.legend()
plt.tight_layout()
plt.show()

## 6. Limpeza e Tratamento de Dados

In [ ]:
# Remover países com muitos dados faltantes (> 50% das features)
threshold_pais = 0.5
missing_por_pais = df_analise[features_numericas].isnull().sum(axis=1) / len(features_numericas)
df_limpo = df_analise[missing_por_pais < threshold_pais].copy()

print(f"Países removidos por excesso de dados faltantes: {len(df_analise) - len(df_limpo)}")
print(f"Países restantes: {len(df_limpo)}")

# Remover features com mais de 40% de dados faltantes
features_validas = []
for col in features_numericas:
    pct_missing = df_limpo[col].isnull().sum() / len(df_limpo) * 100
    if pct_missing < 40:
        features_validas.append(col)
    else:
        print(f"Feature removida: {col} ({pct_missing:.1f}% faltantes)")

# Manter apenas features válidas
colunas_manter = ['Country_Code', 'Country_Name'] + features_validas
df_limpo = df_limpo[colunas_manter].copy()

print(f"\nFeatures mantidas: {len(features_validas)}")

In [ ]:
# Imputação de valores faltantes usando a mediana
for col in features_validas:
    if df_limpo[col].isnull().any():
        mediana = df_limpo[col].median()
        df_limpo[col].fillna(mediana, inplace=True)
        print(f"Imputados {df_limpo[col].isnull().sum()} valores em {col} usando mediana: {mediana:.2f}")

# Verificar se ainda há valores faltantes
print(f"\nValores faltantes restantes: {df_limpo[features_validas].isnull().sum().sum()}")

## 7. Gráfico de Dispersão

In [ ]:
# Funções auxiliares

# padroniza os dados (Z-score) para evitar diferença de escala
def padronizar(X):
    return (X - X.mean(axis=0)) / X.std(axis=0)


# aplica PCA manual e retorna os 2 principais componentes
def pca_2d(X):
    cov = np.cov(X.T)
    eig_vals, eig_vecs = np.linalg.eig(cov)

    # ordena autovalores do maior para o menor
    idx = np.argsort(eig_vals)[::-1]
    eig_vecs = eig_vecs[:, idx]

    # projeta nos 2 primeiros componentes principais
    return np.dot(X, eig_vecs[:, :2])


# Preparação dos dados

# seleciona apenas features válidas
features = [f for f in features_validas if f in df_limpo.columns]

# remove nulos
df_model = df_limpo[features].dropna()

# transforma em matriz numérica
X = df_model.values

# padronização
X = padronizar(X)


# PCA

# aplica redução de dimensionalidade
X_pca = pca_2d(X)

# organiza resultado em DataFrame
df_pca = pd.DataFrame(
    X_pca,
    columns=["PC1", "PC2"],
    index=df_model.index
)


# Gráfico

plt.figure(figsize=(10, 7))

plt.scatter(
    df_pca["PC1"],
    df_pca["PC2"],
    alpha=0.7
)

plt.title("PCA - Projeção dos Países (PC1 vs PC2)")
plt.xlabel("Componente Principal 1")
plt.ylabel("Componente Principal 2")
plt.grid(True)

plt.show()


# Resumo interpretativo

print("""
Resumo:

O PCA reduz os dados para 2 dimensões principais (PC1 e PC2), que concentram a maior parte
da variabilidade do dataset.

PC1 captura a maior variação dos dados, enquanto PC2 captura a segunda direção mais importante
e independente.

Cada ponto representa um país, e a proximidade indica similaridade entre perfis socioeconômicos.

Isso facilita visualizar padrões globais e possíveis agrupamentos entre países.
""")

## 8. Feature Engineering


### Questão 01: Interpretação do PCA

In [ ]:
# 1. PCA 
def pca_manual(df, features):
    X = df[features].values

    # Padronização (Z-score)
    X_std = (X - X.mean(axis=0)) / X.std(axis=0)

    # Matriz de covariância
    cov = np.cov(X_std.T)

    # Autovalores e autovetores
    eig_vals, eig_vecs = np.linalg.eig(cov)

    # Ordenação dos componentes por importância
    idx = np.argsort(eig_vals)[::-1]
    eig_vals, eig_vecs = eig_vals[idx], eig_vecs[:, idx]

    # Variância explicada por cada componente
    var_exp = (eig_vals / np.sum(eig_vals)) * 100

    # Loadings (pesos das variáveis em cada componente)
    loadings = pd.DataFrame(
        eig_vecs[:, :2],
        index=features,
        columns=["PC1", "PC2"]
    )

    return loadings, var_exp


# 2.Interpretação
def exibir_relatorio(loadings, var_exp):

    # Interpretação textual dos componentes
    textos = {
        "PC1": "O PC1 está fortemente associado a indicadores de desenvolvimento socioeconômico, representando um eixo que diferencia países com maior infraestrutura, renda e qualidade de vida daqueles com piores condições sociais. Dessa forma, pode ser interpretado como um índice de desenvolvimento humano e acesso a serviços básicos. Este componente é uma combinação linear das variáveis originais do dataset, formada pelos pesos (loadings) atribuídos a cada variável. As variáveis com maior contribuição são Acesso_Internet, PIB_per_capita_log, Expectativa_Vida, Mortalidade_Infantil e Acesso_Eletricidade. Em conjunto, essas variáveis caracterizam um Índice de Desenvolvimento Econômico e Tecnológico, observa-se ainda que o eixo apresenta associações positivas com Mortalidade_Infantil e associações negativas com Acesso_Internet, PIB_per_capita_log, Expectativa_Vida e Acesso_Eletricidade, indicando um contraste entre países mais e menos desenvolvidos.",
          
        "PC2": "O PC2 está relacionado à estrutura econômica e à dinâmica social dos países, refletindo diferenças no mercado de trabalho, no consumo de recursos e no nível de investimento em setores sociais e produtivos. Dessa forma, pode ser interpretado como um eixo de organização econômica e pressão socioeconômica. Este componente é uma combinação linear das variáveis originais do dataset, formado pelos pesos (loadings) associados a cada variável. As variáveis com maior contribuição são Desemprego, Gasto_Saude_Per_Capita, Consumo_Energia, Taxa_Alfabetizacao e Energia_Renovavel. Em conjunto, essas variáveis caracterizam um eixo de mercado de trabalho e estrutura econômica, observa-se que o componente apresenta associações positivas com Desemprego e Taxa_Alfabetizacao, enquanto possui associações negativas com Gasto_Saude_Per_Capita, Consumo_Energia e Energia_Renovavel, indicando um contraste entre diferentes perfis de organização econômica e social."
    }

    for i, pc in enumerate(["PC1", "PC2"]):

        # Seleciona as 5 variáveis mais importantes (por valor absoluto)
        top = loadings[pc].abs().sort_values(ascending=False).head(5)

        # Cria tabela apenas com os loadings dessas variáveis
        tabela_top = loadings.loc[top.index, [pc]].sort_values(by=pc, ascending=False)

        print(f"\n{'═'*80}")
        print(f" {pc} | Variência explicada: {var_exp[i]:.2f}%")
        print(f"{'═'*80}")

        print("\n Variáveis mais influentes:")
        display(
            tabela_top.style
            .format("{:.4f}")
            .background_gradient(cmap="Blues")
        )

        print("\nAnálise:")
        print(textwrap.fill(textos[pc], width=80))

        print(f"\n{'═'*80}")


loadings, var_exp = pca_manual(df_limpo, features_validas)

exibir_relatorio(loadings, var_exp)

print("\n LOADINGS :")
display(loadings.style.background_gradient(cmap="coolwarm").format("{:.4f}"))

### Questão 02: Criação de Índice de Eficiência de Saúde

In [ ]:
# Criar índice de Eficiência de Saúde
# Eficiência = Expectativa de Vida / Gasto em Saúde per Capita

if 'Expectativa_Vida' in df_limpo.columns and 'Gasto_Saude_Per_Capita' in df_limpo.columns:
    
    # Evitar divisão por zero
    df_limpo['Eficiencia_Saude'] = (
        df_limpo['Expectativa_Vida'] /
        (df_limpo['Gasto_Saude_Per_Capita'] + 1e-6)
    )
    
    # Arredondamento para melhor visualização
    df_limpo['Expectativa_Vida'] = df_limpo['Expectativa_Vida'].round(1)
    df_limpo['Gasto_Saude_Per_Capita'] = df_limpo['Gasto_Saude_Per_Capita'].round(2)
    df_limpo['Eficiencia_Saude'] = df_limpo['Eficiencia_Saude'].round(4)
    
    # Top 10 países mais eficientes
    top_eficientes = (
        df_limpo
        .sort_values(by='Eficiencia_Saude', ascending=False)
        [
            [
                'Country_Name',
                'Expectativa_Vida',
                'Gasto_Saude_Per_Capita',
                'Eficiencia_Saude'
            ]
        ]
        .head(10)
        .reset_index(drop=True)
    )
    top_eficientes.index = top_eficientes.index + 1
    
    # Renomear colunas para melhor visualização
    top_eficientes.columns = [
        'País',
        'Expectativa de Vida',
        'Gasto Saúde per Capita',
        'Eficiência em Saúde'
    ]
    
    print("\nTop 10 Países com Maior Eficiência em Saúde:\n")
    
    # Exibir como tabela organizada
    display(top_eficientes)
    
    print("\n")
    print("Análise:")
    
    print("""
    A Eficiência de Saúde mede quanto um país consegue transformar o investimento em saúde per capita em anos de vida da população. Quanto maior esse índice, maior tende a ser a capacidade do país de gerar bons resultados de saúde com menor custo relativo. Isso não significa necessariamente gastar pouco, mas sim gastar melhor.

    Países mais eficientes geralmente apresentam sistemas públicos de saúde bem estruturados, forte investimento em atenção primária e prevenção, campanhas eficazes de vacinação e controle epidemiológico, boa gestão de recursos públicos, menor desperdício e maior eficiência administrativa, além de educação sanitária e hábitos mais saudáveis da população.

    Por outro lado, países menos eficientes podem apresentar altos gastos com baixa cobertura populacional, forte desigualdade no acesso à saúde, sistemas fragmentados ou pouco integrados, baixa prevenção e maior foco apenas em tratamento, além de ineficiência administrativa e desperdício de recursos.

    Essa métrica é importante porque permite avaliar não apenas quanto um país investe, mas principalmente a qualidade e o retorno social desse investimento.
    """)

    # Adicionar à lista de features válidas
    if 'Eficiencia_Saude' not in features_validas:
        features_validas.append('Eficiencia_Saude')

else:
    print("Features necessárias não disponíveis para cálculo de Eficiência de Saúde")

### Questão 03: Binarização e Categorização (Discretização)

In [ ]:

# Criar variável categórica Nivel_Industrial

def processar_nivel_industrial(df):

    if not {'Exportacoes_PIB', 'Consumo_Energia'}.issubset(df.columns):
        print("Features necessárias não disponíveis")
        return df

    # Normalização Min-Max para evitar que uma variável domine a outra
    def normalize(s):
        return (s - s.min()) / (s.max() - s.min())

    # Criar índice industrial com base nas variáveis normalizadas
    df['Indice_Industrial'] = (
        normalize(df['Exportacoes_PIB']) +
        normalize(df['Consumo_Energia'])
    ).round(4)

    # Classificar os países usando percentil 25 e percentil 75
    labels = ['Baixa', 'Média', 'Alta']

    df['Nivel_Industrial'] = pd.qcut(
        df['Indice_Industrial'],
        q=[0, 0.25, 0.75, 1],
        labels=labels
    )

    # Exibir tabela, gráfico e análise
    exibir_dashboard_industrial(df)

    return df


def exibir_dashboard_industrial(df):

    # Calcular os percentis de corte
    cortes = df['Indice_Industrial'].quantile([0.25, 0.75])

    # Criar tabela com a distribuição dos países
    dist = (
        df['Nivel_Industrial']
        .value_counts()
        .reindex(['Alta', 'Média', 'Baixa'])
        .reset_index()
    )

    dist.columns = [
        'Nível Industrial',
        'Quantidade de Países'
    ]

    print("ANÁLISE E CATEGORIZAÇÃO INDUSTRIAL")
    print("=" * 80)

    # Criar tabela com os pontos de corte dos percentis
    tabela_percentis = pd.DataFrame({
        'Percentil': ['P25', 'P75'],
        'Valor de Corte': [
            round(cortes[0.25], 4),
            round(cortes[0.75], 4)
        ],
        'Classificação': [
            'Baixa Industrialização',
            'Alta Industrialização'
        ]
    })

    print("\nTabela de Percentis:\n")
    display(tabela_percentis)

    print("\nDistribuição dos Países:\n")
    display(dist)

    # Gráfico de barras para visualizar a distribuição
    plt.figure(figsize=(10, 4))

    sns.set_style("whitegrid")

    paleta = {
        "Alta": "#2ecc71",
        "Média": "#f1c40f",
        "Baixa": "#e74c3c"
    }

    sns.barplot(
        data=dist,
        x='Nível Industrial',
        y='Quantidade de Países',
        palette=paleta,
        hue='Nível Industrial',
        legend=False
    )

    plt.title(
        'Distribuição por Nível Industrial',
        fontsize=12,
        fontweight='bold'
    )

    plt.ylabel('Número de Países')
    plt.xlabel('Categoria')
    plt.show()

    # Como essa nova feature ajuda a colorir os pontos no gráfico de PCA

    print("\nAnálise: Como essa nova feature ajuda a colorir os pontos no gráfico de PCA?\n")

    print(f"""
    No gráfico de PCA, essa nova feature é extremamente importante porque ela pode
    ser utilizada como variável de cor, permitindo identificar visualmente
    clusters de países com perfis industriais semelhantes.

    Assim, países com Alta industrialização tendem a se agrupar em regiões próximas,
    países com Baixa industrialização podem formar outros grupos distintos, e os de
    nível Médio geralmente aparecem como uma transição entre esses grupos.

    Isso torna a análise multivariada muito mais clara, facilita a interpretação
    dos componentes principais e melhora significativamente a visualização dos
    padrões estruturais de desenvolvimento industrial.
    """)


# Execução principal
df_limpo = processar_nivel_industrial(df_limpo)

### Questão 04: Normalização Condicional

In [ ]:

# PCA 2D manual
def pca_2d(X):
    # reduz assimetria com log
    X = np.log1p(np.abs(X))

    # padroniza (média 0, desvio 1)
    X = (X - X.mean(axis=0)) / X.std(axis=0)

    cov = np.cov(X.T)
    eig_vals, eig_vecs = np.linalg.eig(cov)

    # ordena componentes
    idx = np.argsort(eig_vals)[::-1]
    eig_vals = eig_vals[idx]
    eig_vecs = eig_vecs[:, idx]

    X_pca = X @ eig_vecs[:, :2]
    return X_pca, eig_vals


# limpa duplicadas
df_limpo = df_limpo.loc[:, ~df_limpo.columns.duplicated()].copy()

# pega só features válidas
features = [
    c for c in features_validas
    if c in df_limpo.columns and c != 'Populacao_Total'
]

# base sem nulos
df_base = df_limpo[features + ['Populacao_Total']].dropna().reset_index(drop=True)

# detecta outliers usando per capita bruto
pop = df_base['Populacao_Total'].to_numpy(dtype=float) / 1e6
X_percapita_raw = df_base[features].to_numpy(dtype=float) / pop[:, None]

# remove extremos (3 desvios padrão)
mask = (np.abs(stats.zscore(X_percapita_raw)) < 3).all(axis=1)
df_final = df_base[mask].reset_index(drop=True)

# dados antes
X_antes = df_final[features].to_numpy(dtype=float)

# dados depois (per capita)
pop_final = df_final['Populacao_Total'].to_numpy(dtype=float) / 1e6
X_depois = X_antes / pop_final[:, None]

# PCA antes e depois
X_pca_antes, eig_antes = pca_2d(X_antes)
X_pca_depois, eig_depois = pca_2d(X_depois)

var_antes = eig_antes / eig_antes.sum()
var_depois = eig_depois / eig_depois.sum()

# gráfico comparativo
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(X_pca_antes[:, 0], X_pca_antes[:, 1], alpha=0.6)
axes[0].set_title("PCA ANTES (absoluto)")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")
axes[0].grid(True)

axes[1].scatter(X_pca_depois[:, 0], X_pca_depois[:, 1], alpha=0.6, color='orange')
axes[1].set_title("PCA DEPOIS (per capita + filtro)")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
axes[1].grid(True)

plt.tight_layout()
plt.show()

# resultado
print(f"Variância PC1 antes: {var_antes[0]:.4f}")
print(f"Variância PC1 depois: {var_depois[0]:.4f}")

# explicação final
print("""
A variância do primeiro componente aumentou após a normalização per capita. Antes da transformação, o PCA era fortemente influenciado
por diferenças de escala entre os países, principalmente relacionadas à população e aos valores absolutos das variáveis. Isso fazia
com que a variação dos dados fosse mais distribuída entre os componentes principais, já que múltiplos fatores de magnitude coexistiam
simultaneamente.
Após a normalização per capita e a remoção de outliers, as variáveis passaram a representar a intensidade dos indicadores por habitante,
reduzindo o efeito de escala e tornando os dados mais comparáveis entre países. Com isso, um padrão estrutural mais consistente passou a
se destacar, fazendo com que o primeiro componente principal explicasse uma maior proporção da variância total. Na prática, isso indica
que o PCA passou a capturar melhor o nível geral de desenvolvimento socioeconômico dos países.

Por exemplo, antes da normalização, países com grande população e altos valores absolutos tendiam a dominar a estrutura do PCA apenas por
magnitude. Após a normalização, países menores mas mais eficientes podem se aproximar de países maiores, pois a comparação passa a ser ba
seada em intensidade e não em tamanho. Assim, o PCA deixa de refletir principalmente escala populacional e passa a representar diferenças estruturais reais entre os países.
""")


### Questão 05: Tratamento de Skewness (Assimetria)

In [ ]:
# Análise do PIB per capita e impacto do log

if 'PIB_per_capita' in df_limpo.columns:

    # gráfico comparativo
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # dados base
    dados = df_limpo['PIB_per_capita'].dropna()

    # distribuição original
    axes[0].hist(dados, bins=30, edgecolor='black', alpha=0.7)
    axes[0].set_title('PIB per capita - ORIGINAL')
    axes[0].set_xlabel('PIB per capita')
    axes[0].set_ylabel('Frequência')

    # média e mediana
    axes[0].axvline(dados.mean(), color='red', linestyle='--', label='Média')
    axes[0].axvline(dados.median(), color='green', linestyle='--', label='Mediana')
    axes[0].legend()

    # skew original
    skew_original = dados.skew()

    # transformação log
    df_limpo['PIB_per_capita_log'] = np.log1p(df_limpo['PIB_per_capita'])

    dados_log = df_limpo['PIB_per_capita_log'].dropna()

    # distribuição após log
    axes[1].hist(dados_log, bins=30, edgecolor='black', alpha=0.7, color='orange')
    axes[1].set_title('PIB per capita - LOG')
    axes[1].set_xlabel('log(PIB per capita)')
    axes[1].set_ylabel('Frequência')

    # média e mediana
    axes[1].axvline(dados_log.mean(), color='red', linestyle='--', label='Média')
    axes[1].axvline(dados_log.median(), color='green', linestyle='--', label='Mediana')
    axes[1].legend()

    # skew após log
    skew_log = dados_log.skew()

    plt.tight_layout()
    plt.show()

    # resultados numéricos
    print(f"\nSkewness original: {skew_original:.4f}")
    print(f"Skewness após log: {skew_log:.4f}")

    # mantém feature para análise
    if 'PIB_per_capita' in features_validas:
        features_validas.remove('PIB_per_capita')

    if 'PIB_per_capita_log' not in features_validas:
        features_validas.append('PIB_per_capita_log')

else:
    print("PIB_per_capita não disponível")


# explicação final

print("""
A distribuição do PIB per capita apresenta forte assimetria à direita, o que indica que poucos países extremamente ricos puxam a escala dos dados
e distorcem a análise. Após a aplicação da transformação logarítmica, essa assimetria é reduzida, tornando a distribuição mais equilibrada e apro
ximando o comportamento dos dados de uma forma mais linear. Isso é importante porque o PCA é sensível a escalas e variações extremas.
Posto isso, o log ajuda a reduzir o impacto de outliers, evita que países muito ricos dominem o primeiro componente principal e permite que o mo-
delo capture melhor diferenças entre países intermediários.Em termos gerais, a transformação faz com que o PCA passe a analisar variações relati-
vas entre os países em vez de ser influenciado apenas por diferenças absolutas muito grandes.
""")